In [64]:
import pandas as pd 
import numpy as np

# Loading data into my dataframe
df = pd.read_csv("diabetic_data.csv")

# Getting initial info for the data
df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 50 columns):
 #   Column                    Non-Null Count   Dtype 
---  ------                    --------------   ----- 
 0   encounter_id              101766 non-null  int64 
 1   patient_nbr               101766 non-null  int64 
 2   race                      101766 non-null  object
 3   gender                    101766 non-null  object
 4   age                       101766 non-null  object
 5   weight                    101766 non-null  object
 6   admission_type_id         101766 non-null  int64 
 7   discharge_disposition_id  101766 non-null  int64 
 8   admission_source_id       101766 non-null  int64 
 9   time_in_hospital          101766 non-null  int64 
 10  payer_code                101766 non-null  object
 11  medical_specialty         101766 non-null  object
 12  num_lab_procedures        101766 non-null  int64 
 13  num_procedures            101766 non-null  int64 
 14  num_

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


### Handling the Nulls

* Replacing all '?' with actual NaN values across the entire dataframe

In [65]:
# Making a copy of dataframe and manipulating the copy instead of original
df_copy = df.copy()
df_copy.replace('?', np.nan, inplace=True)

# Checking the total amount of cells changed (i.e. how much missing data there was)
print(df_copy.isnull().sum() / len(df_copy) * 100)

encounter_id                 0.000000
patient_nbr                  0.000000
race                         2.233555
gender                       0.000000
age                          0.000000
weight                      96.858479
admission_type_id            0.000000
discharge_disposition_id     0.000000
admission_source_id          0.000000
time_in_hospital             0.000000
payer_code                  39.557416
medical_specialty           49.082208
num_lab_procedures           0.000000
num_procedures               0.000000
num_medications              0.000000
number_outpatient            0.000000
number_emergency             0.000000
number_inpatient             0.000000
diag_1                       0.020636
diag_2                       0.351787
diag_3                       1.398306
number_diagnoses             0.000000
max_glu_serum               94.746772
A1Cresult                   83.277322
metformin                    0.000000
repaglinide                  0.000000
nateglinide 

### Data Cleaning & Clinical Signal Preservation
Healthcare datasets require careful handling of null values, as "missing" data often contains critical signals. 

* **Administrative Data Strategy:** The `weight` feature was removed entirely due to >96% missing data, making computation unreliable. Aswell, missing values in `payer_code` (insurance) and `medical_specialty` were re-categorized as `'Unknown'`. 
* **Clinical Feature Engineering:** Blood sugar diagnostics (`A1Cresult` and `max_glu_serum`) exhibited >80% missingness. Rather than dropping these columns, the nulls were imputed as `'Not Tested'`.

In [66]:
# 1. Dropping the unnecessary weight column
df_copy = df_copy.drop(columns='weight')

# 2. Fill the missing administrative columns
df_copy['payer_code'] = df_copy['payer_code'].fillna('Unknown')
df_copy['medical_specialty'] = df_copy['medical_specialty'].fillna('Unkown')

# 3. Fill the missing lab test columns to preserve clinical signal
df_copy['max_glu_serum'] = df_copy['max_glu_serum'].fillna('Not Tested')
df_copy['A1Cresult'] = df_copy['A1Cresult'].fillna('Not Tested')

# Sanity Check
df_copy.head()

,encounter_id,patient_nbr,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),6,25,1,1,Unknown,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),1,1,7,3,Unknown,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),1,1,7,2,Unknown,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),1,1,7,2,Unknown,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),1,1,7,1,Unknown,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


### Target Variable Standardization (HRRP Compliance)
The hospital is financially penalized for patients who are readmitted within a 30-day window. To prepare this data for SQL analysis and predictive modeling, we must convert the categorical `readmitted` column into a binary target variable (`target_readmitted`):
* **`1` (Penalty):** Patient was readmitted in `< 30` days.
* **`0` (Compliant):** Patient was readmitted in `> 30` days, or `NO` readmission occurred.

In [67]:
df_copy['target_readmitted'] = np.where((df_copy['readmitted'] == '>30') | (df_copy['readmitted'] == 'NO'), 0, 1)

# Quick sanity check to see our readmission breakdown 
print(df_copy['target_readmitted'].value_counts())

target_readmitted
0    90409
1    11357
Name: count, dtype: int64


### Feature Engineering (Creating Medical Cohorts)
We will be creating Medical Cohorts, we will group patients into th following buckets: 

* 1-10 Meds
* 11-20 Meds
* 21-30 Meds
* 30+ Meds

We will use these cohorts to later test this hypothesis: **"Do patients prescribed massive amounts of medication bounce back to the hospital faster?"**

In [68]:
bins = [0, 10, 20, 30, np.inf]
labels = ['1-10 Meds', '11-20 Meds', '21-30 Meds', '30+ Meds']

medical_cohort =pd.cut(df_copy['num_medications'], bins = bins, labels = labels)

df_copy['medical_cohort'] = medical_cohort
df_copy.head()

,encounter_id,patient_nbr,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,...,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted,target_readmitted,medical_cohort
0,2278392,8222157,Caucasian,Female,[0-10),6,25,1,1,Unknown,...,No,No,No,No,No,No,No,NO,0,1-10 Meds
1,149190,55629189,Caucasian,Female,[10-20),1,1,7,3,Unknown,...,No,No,No,No,No,Ch,Yes,>30,0,11-20 Meds
2,64410,86047875,AfricanAmerican,Female,[20-30),1,1,7,2,Unknown,...,No,No,No,No,No,No,Yes,NO,0,11-20 Meds
3,500364,82442376,Caucasian,Male,[30-40),1,1,7,2,Unknown,...,No,No,No,No,No,Ch,Yes,NO,0,11-20 Meds
4,16680,42519267,Caucasian,Male,[40-50),1,1,7,1,Unknown,...,No,No,No,No,No,Ch,Yes,NO,0,1-10 Meds


### Dropping useless columns and exporting the file
In SQL, aggregations don't need random ID numbers. So we will drop the `encounter_id` and `patient_nbr` columns from df_copy. Then we will be exporting our cleaned data into `cleaned_hospital_readmissions.csv` so we can query it in SQL. 

In [69]:
df_copy = df_copy.drop(columns=['encounter_id', 'patient_nbr'])
df_copy.head()

df_copy.to_csv('cleaned_hospital_readmissions.csv', index=False)